# 10 · CRISP-DM: iterating between data preparation and modelling
Adapted from the tutorial notebook *Preprocessing_and_CRISP_DM* (California housing, regression) to the DSC
(EPC label, 8-class classification). The tutorial's message: preprocessing choices are **tested**, not assumed.
Each choice is fitted on the training data and judged by its score on unseen data. Here that is
**1-notch accuracy (the official DSC metric) and macro-F1 on the validation set**; the test set stays untouched for the final models (notebooks 03–08).

Run this notebook after `02_data_preparation`. It does not overwrite anything: it only compares choices.
If a choice clearly wins, copy it into `02_data_preparation` and rerun the model notebooks.

## 1 · Business understanding
**Who would use an EPC prediction, and for what?**
- **Banks:** estimate the energy label of a house used as mortgage collateral when no certificate is on file
  (energy-efficient loans, climate-risk reporting on the mortgage book).
- **Governments and renovation programmes:** find likely energy-poor buildings (E/F/G) to target renovation support.
  In Flanders, buyers of E/F homes must renovate to at least D within a set period.
- **Real-estate platforms:** show an estimated label for listings without a certificate.

**What does a mistake cost?** The labels are ordered: predicting D for a C building is a small error,
predicting A for a G building is a large one. That is why the project reports ordinal MAE and within-one
accuracy next to macro-F1, and E/F/G recall for the renovation use case.

**What the data says about the business:** the label scale differs per region (no G in Flanders, A+ almost only
in Wallonia), so a single model must learn three regional scales, or one model per region could be considered.

## 2 · Data understanding (recap of notebook 01)
| Finding | Preparation choice to test here |
|---|---|
| Size variables (surface, volume, price) are strongly right-skewed | scale as is vs. log vs. quantile transform |
| ~10 text columns with 50–1000 categories (municipality, ZIP, regional planning, destination) | one-hot (rare categories grouped) vs. target encoding vs. dropping them |
| Rare categories | `min_frequency` 100 vs. 500 |
| 115k training rows | does a model still improve with more data? (learning curve) |

In [ ]:
import sys
from pathlib import Path

# find the project root (the folder that contains src/dsc_utils.py) and import the shared helpers
root = Path.cwd()
while not (root / "src" / "dsc_utils.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dsc_utils import *
import time
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, QuantileTransformer, FunctionTransformer, TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

d = load_prepared()
data = d["data"]
y_train, y_val = d["y_train"], d["y_val"]
X_tr, X_va = data.iloc[d["indices_train"]], data.iloc[d["indices_val"]]

# experiments run on samples so each one takes about a minute; all are fitted on TRAIN rows only
X_tr_s, y_tr_s = stratified_sample(X_tr, y_train, 30000)
X_va_s, y_va_s = stratified_sample(X_va, y_val, 10000)

cat_cols = data.select_dtypes(exclude="number").columns.tolist()
num_cols = data.select_dtypes(include="number").columns.tolist()
high_card = [c for c in cat_cols if X_tr[c].nunique() > 50]
low_card = [c for c in cat_cols if c not in high_card]
skew = X_tr[num_cols].skew(numeric_only=True)
skewed = skew[skew.abs() >= 1].index.tolist()
print(len(num_cols), "numeric (", len(skewed), "skewed ) |", len(low_card), "low-cardinality text |",
      len(high_card), "high-cardinality text:", high_card)

## 3 · Data preparation: one function, several choices
`make_preprocessor()` builds the pipeline from a few switches. Every variant below is one call with different switches.

In [ ]:
def signed_log(x):
    return np.sign(x) * np.log1p(np.abs(x))

def make_preprocessor(numeric="standard", high_card_mode="onehot", min_frequency=100):
    """numeric: 'standard' | 'log' | 'quantile' (applied to the skewed columns)
       high_card_mode: 'onehot' | 'target' (target encoding) | 'drop'"""
    impute = ("imputer", SimpleImputer(strategy="median"))
    skew_steps = {"standard": [impute, ("scaler", StandardScaler())],
                  "log": [impute, ("log", FunctionTransformer(signed_log)), ("scaler", StandardScaler())],
                  "quantile": [impute, ("quantile", QuantileTransformer(output_distribution="normal",
                                                                       n_quantiles=1000, random_state=0))]}
    other_num = [c for c in num_cols if c not in skewed]
    onehot = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=min_frequency))])
    parts = [("num_skewed", Pipeline(skew_steps[numeric]), skewed),
             ("num_other", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), other_num),
             ("cat_low", onehot, low_card)]
    if high_card_mode == "onehot":
        parts.append(("cat_high", onehot, high_card))
    elif high_card_mode == "target":
        parts.append(("cat_high", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                            ("target", TargetEncoder(target_type="multiclass", random_state=0))]), high_card))
    return ColumnTransformer(parts)

def run_variant(name, **switches):
    """Fit the preprocessor on the training sample, then score two fast models on the validation sample."""
    t = time.time()
    pre = make_preprocessor(**switches)
    xt = pre.fit_transform(X_tr_s, y_tr_s)        # y is only used by target encoding
    xv = pre.transform(X_va_s)
    lr = LogisticRegression(C=0.1, max_iter=1000).fit(xt, y_tr_s)
    dt = DecisionTreeClassifier(criterion="entropy", min_samples_leaf=50, random_state=0).fit(xt, y_tr_s)
    row = {"variant": name, "n_features": xt.shape[1],
           "LR_macro_F1": macro_f1(y_va_s, lr.predict(xv)),
           "DT_macro_F1": macro_f1(y_va_s, dt.predict(xv)),
           "LR_1notch": one_notch_accuracy(y_va_s, predict_one_notch(class_scores(lr, xv))),
           "DT_1notch": one_notch_accuracy(y_va_s, predict_one_notch(class_scores(dt, xv))),
           "seconds": round(time.time() - t)}
    print(row)
    return row

## 4 · Modelling to judge the preparation choices

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

experiments = [
    run_variant("baseline (as in 02)"),
    run_variant("skewed: log", numeric="log"),
    run_variant("skewed: quantile", numeric="quantile"),
    run_variant("rare categories: min_frequency 500", min_frequency=500),
    run_variant("high-cardinality: target encoding", high_card_mode="target"),
    run_variant("high-cardinality: dropped", high_card_mode="drop"),
]
results_prep = pd.DataFrame(experiments).set_index("variant")
results_prep.round(4)

In [ ]:
results_prep[["LR_1notch", "DT_1notch", "LR_macro_F1", "DT_macro_F1"]].plot(kind="barh", figsize=(8, 5))
plt.xlabel("validation score (1-notch accuracy = official DSC metric)"); plt.title("Effect of preparation choices"); plt.show()

**How to read it:** logistic regression is sensitive to scaling and encoding; a decision tree is not (it only
uses the order of values). A choice that helps LR but not DT matters for linear models, SVM and kNN only.
A difference below about 0.005 is within noise for a 10k validation sample.

## 5 · How much training data does a model need? (learning curve)
The tutorial's TODO "try different sizes for the train-test split", done systematically. If validation macro-F1
stops rising well before the full training set, the 64/16/20 split costs nothing.

In [ ]:
x_train_full, x_val_full = d["x_train"], d["x_val"]
sizes = [5000, 10000, 20000, 40000, 80000, len(y_train)]
curve = []
for n in sizes:
    xs, ys = stratified_sample(x_train_full, y_train, n)
    m = DecisionTreeClassifier(criterion="entropy", min_samples_leaf=50, random_state=0).fit(xs, ys)
    curve.append(macro_f1(y_val, m.predict(x_val_full)))
    print(n, round(curve[-1], 4))

plt.plot(sizes[:len(curve)], curve, marker="o"); plt.xscale("log")
plt.xlabel("training rows"); plt.ylabel("validation macro-F1"); plt.title("Learning curve (decision tree)")
plt.show()

## 6 · Several models at once with cross-validation (the tutorial's MODELS / HYPERPARAM pattern)
A quick screening on a 20k sample with 3-fold stratified cross-validation, using the baseline preparation.
The full tuning per model is in notebooks 03–08.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

xs, ys = stratified_sample(d["x_train"], y_train, 20000)
MODELS = {"dt": DecisionTreeClassifier, "lr": LogisticRegression, "rf": RandomForestClassifier}
HYPERPARAM = {
    "dt": {"criterion": ["entropy"], "min_samples_leaf": [20, 50, 100], "random_state": [0]},
    "lr": {"C": [0.01, 0.1, 1], "max_iter": [1000]},
    "rf": {"n_estimators": [100], "min_samples_leaf": [5, 20], "max_samples": [0.5], "random_state": [0]},
}
screening = []
for name, model in MODELS.items():
    gs = GridSearchCV(model(), HYPERPARAM[name], scoring="f1_macro",
                      cv=StratifiedKFold(3, shuffle=True, random_state=42), n_jobs=2)
    gs.fit(xs, ys)
    screening.append({"model": name, "best_params": gs.best_params_, "cv_macro_F1": round(float(gs.best_score_), 4),
                      "val_macro_F1": round(float(macro_f1(y_val, gs.best_estimator_.predict(d["x_val"]))), 4)})
    print(screening[-1])
pd.DataFrame(screening)

## 7 · Evaluation and deployment (CRISP-DM)
**Write down** (for the report): the preparation choices that improved validation macro-F1, the final model and
its hyperparameters (notebooks 03–08), and its **test** scores from `outputs/DSC_results.csv`.

**Deployment questions to answer in the report:**
- The scoring file was checked to look like the training data (notebook 01, Step 7). If it had not, test scores would overstate real performance.
- EPC rules change over time (new calculation methods, regional reforms), so a deployed model would need monitoring and periodic retraining.
- Predictions near a label boundary are uncertain; a bank or government would want the class probabilities, not just the letter.